# Mini Transformer
**Cours NLP - CY Tech**

---

**By: Alexandre Huet**

---

# TP1 - Du texte brut au premier modèle bigramme

### Libraries required

In [ ]:
import os
from urllib.request import urlretrieve

import torch
import torch.nn as nn
from torch.nn import functional as F

### Étape 1: télécharger et inspecter le corpus

In [ ]:
import urllib.request
import os


URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

texte = urllib.request.urlopen(URL).read().decode("utf-8")

print(f"{len(texte):,} caractères")
print(f"{len(texte.splitlines()):,} lignes")
print("\n 1000 premiers caractères \n")
print(texte[:1000])

1,115,394 caractères
40,000 lignes

 1000 premiers caractères 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods kno

**La longueur du corpus:** Le corpus contient environ 1,115,394 caractères.

**Les retours à la ligne:** On voit des retours à la ligne qui séparent les noms de personne de leurs répliques et qui séparent aussi les répliques entre elles.

**La ponctuation:** On trouve tous les ponctuation classique comme des virgules, des points, des points d'interrogation et etc.

**Les majuscules et minuscules:** Le texte contient une melange des majuscules et des minuscules. Il sont traitées comme des caractères différents dans le vocabulaire.

### Questions d’interprétation

##### Q1: Quelle est ici l'unité d'un token?
- Ici l'unité d'un token est un caractère donc une lettre, un espace, un signe de ponctuation ou un retour à la ligne et pas un mot.

##### Q2: Pourquoi préciser encoding="utf-8"?

- On précise utf-8 pour être sûr que le fichier soit lu de la même façon sur toutes les machines. Sinon certains caractères spéciaux pourraient être mal décodés.

##### Q3: Le fichier estil déjà séparé en exemples?


- Non, le fichier est une seule longue chaîne de caractères continue. Il faut donc que nous construisions nous-mêmes les exemples. C'est-à-dire les couples contexte et cible, à partir de ce texte.

### Étape 2: passer des caractères aux entiers

In [ ]:
chars = sorted(list(set(texte)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

message = "hii there"
print(encode(message))
print(decode(encode(message)))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


### Questions d’interprétation

##### Q1: Pourquoi trier chars?

- On trie les chars pour garantir un ordre stable et reproductible. Si on ne trie pas les chars, l'ordre des caractères pourrait changer d'une exécution à l'autre ce qui donnerait des identifiants différents pour les mêmes caractères.

##### Q2: Que garantit le test decode(encode(message)) == message ?


- Ce test garantit que encode et decode sont bien les fonctions inverses l'une de l'autre. Donc aucune information n'est perdue lors de la conversion du texte en entiers puis à nouveau en texte.


###  Étape 3: fabriquer les prédictions attendues

In [ ]:
data = torch.tensor(encode(texte), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(context, "->", target)

tensor([18]) -> tensor(47)
tensor([18, 47]) -> tensor(56)
tensor([18, 47, 56]) -> tensor(57)
tensor([18, 47, 56, 57]) -> tensor(58)
tensor([18, 47, 56, 57, 58]) -> tensor(1)
tensor([18, 47, 56, 57, 58,  1]) -> tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) -> tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> tensor(58)


### Questions d’interprétation

##### Q1: Combien de prédictions produit ce bloc?

- Ce bloc produit 8 prédictions. Une pour chaque position t dans range block_size.

##### Q2: Pourquoi ne pas mélanger train et validation?

- On ne mélange pas train et validation car le texte est séquentiel. Mélanger casserait l'ordre du texte et pourrait faire fuir des informations entre les deux ensembles.

##### Q3: Pour t = 3, quels sont le contexte et la cible?

- Pour t = 3, le contexte est x[:4] donc c'est les 4 premiers caractères de la fenêtre et la cible est y[3], le caractère qui suit directement ce contexte.

### Étape 4 : échantillonner un mini-batch

In [ ]:
batch_size = 4
block_size = 8

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(
        len(source) - block_size, (batch_size,)
    )
    x = torch.stack([
        source[i : i + block_size] for i in starts
    ])
    y = torch.stack([
        source[i + 1 : i + block_size + 1] for i in starts
    ])
    return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape)

torch.Size([4, 8]) torch.Size([4, 8])


### Questions d’interprétation

##### Q1: Pourquoi retirer block_size à la borne de tirage?

- On retire block_size à la borne de tirage pour éviter qu'une position de départ tirée aléatoirement soit trop proche de la fin du texte.

##### Q2: Que changerait B = 32?

- Avec B = 32, on traiterait 32 séquences indépendantes en parallèle au lieu de 4, ce qui rendrait l'entraînement plus rapide et les gradients plus stables mais demanderait beaucoup plus de mémoire.

##### Q3: Que changerait T = 64?

- Avec T = 64, chaque séquence contiendrait 64 caractères de contexte au lieu de 8. En effet, ca permettrait au modèle de voir plus de contexte mais rendrait chaque calcul plus coûteux.



### Étape 5 : une table produit les logits

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(
            vocab_size, vocab_size
        )

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

### Questions d’interprétation

##### Q1: Pourquoi C × C ici?

- C'est C × C car ce modèle bigramme ne regarde qu'un seul caractère précédent. Il faut donc, pour chacun des C caractères possibles en entrée, un score pour chacun des C caractères possibles en sortie.

##### Q2: Où retrouvera-t-on demb et la couche lm_head dans le TP 4?

- Dans le TP 4, la table d'embedding ne produira plus directement des logits mais un vecteur de dimension d_emb par token. Ce vecteur passera ensuite par les blocs d'attention et de MLP puis la couche lm_head convertira le résultat final en logits sur le vocabulaire.


### Étape 6: générer, puis apprendre

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = BigramLanguageModel(vocab_size)

context = torch.zeros((1, 1), dtype=torch.long)
print("Avant entraînement:")
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))

Avant entraînement:

qW wEP$Btoej3EyTBJRquaqVRnrl?PPXCEIsWNnFZv?$p?hkSD.WN
McBPPbyTeyS&ddYi 
KRjGidVVjxDwh$oX!ScYqu$h';dt


### Étape 7: Entraîner

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(100):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Loss finale:", loss.item())

print("Après entraînement:")
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))

Loss finale: 4.959466934204102
Après entraînement:

vnpXwZJMWg3pgiw'PwYpNvokqCPfJk gzScikoWHXhQn:QtbYoNNHhVcDi C
hVQQExIqfgr'gIX bakSRIdYSJkejhk;NesZ,iD


## Pour 1000 étapes.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(1000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Loss finale:", loss.item())

print("Après entraînement:")
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))

Loss finale: 3.9883317947387695
Après entraînement:

wpgcXk;xQ&PMDGt,zLX BhS
KDpm
qVnHqpWb&knwhad;NauRP!lRmZintmZzjsfp'oFq';IPXDOGNDycy'tako3.xbSdKgl
LUj


### Bilan du TP 1

##### Q1: Pourquoi ne garde-t-on que le dernier pas de temps pendant la génération?

- On ne garde que le dernier pas de temps parce que pendant la génération, on ajoute un seul nouveau caractère à la fois. Seule la prédiction pour la position la plus récente est utile pour choisir ce caractère suivant.

##### Q2: Pourquoi aplatit-on B × T pendant la loss?

- On aplatit B × T car cross_entropy attend une prédiction par exemple pas une prédiction par séquence. Aplatir permet de traiter chaque position de chaque séquence comme un exemple de classification indépendant.

##### Q3: Comparez la loss et la structure du texte avant/après 100 étapes, puis avec 1000 étapes

- à 100 étapes la loss diminue de 4.57 et à 3.48 à 1 000 étapes. Ca montre que le modèle apprend progressivement des régularités statistiques entre caractères. Cependant, le texte généré reste très proche du bruit dans les deux cas, sans mots reconnaissables ni structure grammaticale. En effet, ca s'explique par la limite structurelle du modèle bigramme. Il ne regarde qu'un seul caractère précédent et il ne peut donc jamais capturer autre chose que des statistiques très locales quel que soit le nombre d'étapes d'entraînement.

# TP2 - Construire l’auto attention pas à pas

### Étape 1: agréger par produit matriciel

In [ ]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)

b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


### Questions d’interprétation

##### Q1: Sans Python, calculez c2. Pourquoi la somme de chaque ligne de a vaut-elle 1?

- c2 est la moyenne des trois lignes de b. Donc (b0 + b1 + b2) / 3, car la ligne 2 de a vaut [1/3, 1/3, 1/3]. Chaque ligne de a somme à 1 car elle a été normalisée en divisant par la somme de ses valeurs. En effet, ca garantit que c est bien une moyenne pondérée et pas une simple somme.

###  Étape 2: moyenner uniquement le passé

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, dim=0)

### Questions d’interprétation

##### Q1: Donnez les formes de xprev pour (t=0), (t=3) et (t=7). moyenne porte-t-elle sur dim=0?

- xprev a pour forme (1, C) à t=0, (4, C) à t=3, et (8, C) à t=7. Parce que xprev contient toujours t+1 vecteurs. On moyenne sur dim=0 car c'est la dimension des positions. On veut réduire le nombre de positions à 1 tout en gardant les C features intactes.

### Étapes 3: vectoriser (matricielle)

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
print(torch.allclose(xbow, xbow2, atol=1e-6))

True


### Étapes 4: vectoriser (masque + softmax)


In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(xbow, xbow3, atol=1e-6))

True


### Questions d’interprétation

##### Q1: Pourquoi wei @ x fonctionne-t-il avec une matrice (T , T) et un tenseur (B, T, C)?

- wei @ x fonctionne car PyTorch applique automatiquement la même matrice (T,T) à chacun des B batches grâce au broadcasting.

##### Q2: Que devient Softmax aux emplacements contenant −∞?

- Aux emplacements contenant −∞, Softmax donne exactement 0. Car exp(−∞) = 0. Ces positions futures reçoivent donc un poids nul.


###  Étape 5: calculer des poids d’attention

In [ ]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      #(B,T,head_size)
q = query(x)    #(B,T,head_size)
wei = q @ k.transpose(-2, -1)   #(B,T,T)
wei = wei * head_size**-0.5
wei = wei.masked_fill(tril[:T, :T] == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v   #(B,T,head_size)

### Questions d’interprétation

##### Q1: Quelle ligne de wei[0] correspond au dernier token?

- La dernière ligne de wei[0] (ligne T-1) correspond au dernier token parce que chaque ligne i contient les poids d'attention du token i vers tous les tokens précédents.

##### Q2: Pourquoi la sortie mélanget-elle (v), et non directement (k)?

- La sortie mélange v et non k car k sert uniquement à mesurer la pertinence par rapport à la query. Alors que v contient l'information réellement transmise une fois cette pertinence calculée.

### Étape 6: stabiliser Softmax

In [ ]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

raw = q @ k.transpose(-2, -1)
scaled = raw * head_size**-0.5

print(raw.var(), scaled.var())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

tensor(19.8478) tensor(1.2405)
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


### Bilan du TP2

#### Vérifiez la somme des lignes de wei, l'absence de poids vers le futur et la forme finale. Expliquez le rôle distinct de Q, K et V.

- La somme de chaque ligne de wei vaut bien 1.0 pour les 8 positions. Ce qui confirme que Softmax produit une vraie distribution de probabilité sur chaque ligne. La matrice wei[0] montre bien des zéros au-dessus de la diagonale (par exemple la ligne 0 vaut [1., 0., 0., 0., 0., 0., 0., 0.]). En effet, ca confirme que le masque causal empêche tout accès au futur. La forme finale de out est torch. Size([4, 8, 16]) soit (B, T, head_size) comme attendu.

- Q, K et V ont des rôles distincts: Q représente ce que le token cherche, K représente ce qu'un token peut offrir pour être trouvé et V représente l'information réellement transmise une fois qu'un token a été jugé pertinent par le produit Q·K.

## TP 3 - Stabiliser puis transformer chaque token

 ## Étape 1: coder LayerNorm

In [ ]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(dim=-1, keepdim=True)
        xvar = x.var(
            dim=-1, keepdim=True, unbiased=False
        )
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta

## Questions d’interprétation

##### Sur quel axe normaliser ? À quoi servent ε, γ et β ?


- Q1: On normalise sur l'axe des features (dim=-1). Pour chaque token indépendamment, à travers ses propres composantes. ε évite une division par zéro si la variance est nulle. γ et β sont deux paramètres appris qui permettent au modèle de réajuster l'échelle et le décalage après la normalisation.

## Étape 2: choisir l’axe de normalisation

In [ ]:
torch.manual_seed(1337)
module = LayerNorm1d(100)

x = torch.randn(32, 100)
x = module(x)

print(x[:, 0].mean(), x[:, 0].std())
print(x[0, :].mean(), x[0, :].std())

tensor(0.1476) tensor(0.8847)
tensor(-2.2650e-08) tensor(1.0050)


## Questions d’interprétation

##### Q1: Prédisez quelle ligne donnera environ (0, 1) avant d’exécuter le code.

- Je pense que c'est x[0, :] qui donnera environ (0, 1) parce que LayerNorm normalise chaque token à travers ses propres features. Pas une feature donnée à travers tout le batch. x[:, 0] n'a donc aucune raison d'avoir une moyenne de 0 ou un écart type de 1.



## pour verifier

In [ ]:
print(torch.allclose(module(x), nn.LayerNorm(100)(x), atol=1e-6))

True


## Étape 3: construire le réseau feed-forward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)


ffwd = FeedForward(100)
delta = ffwd(torch.randn(32, 100))
print(delta.shape)

torch.Size([32, 100])


## Questions d’interprétation

##### Q1: Quelle proportion d’activations est positive ? Pourquoi revenir à C?


- Quelle proportion d’act
Après ReLU, environ la moitié des activations sont positives et l'autre moitié sont ramenées à 0, car les valeurs avant ReLU sont réparties de façon à peu près symétrique autour de 0 avec des poids initialisés aléatoirement. On revient à C pour que le résultat delta ait la même forme que le vecteur d'embedding initial, ce qui permet de l'ajouter à ce vecteur via la connexion résiduelle.

## Étapes 4 et 5: ajouter le résiduel

In [ ]:
n_embd = 32
B, T = 4, 8

layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)

x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))
x_out = x + delta

print(x.shape)
print(delta.shape)
print(x_out.shape)

torch.Size([4, 8, 32])
torch.Size([4, 8, 32])
torch.Size([4, 8, 32])


## Bilan du TP 3

##### Q1: Quelle dimension LayerNorm conserve-t-elle?

- LayerNorm conserve la dimension d'embedding. La forme du tenseur en entrée et en sortie reste identique. LayerNorm ne fait que recentrer et redimensionner les valeurs à l'intérieur de chaque token.

##### Q2: Le MLP mélange-t-il les tokens?

- Non, le MLP ne mélange pas les tokens. La même transformation est appliquée indépendamment à chaque token. Sans aucune interaction entre les positions. Seul le mécanisme d'attention mélange l'information entre les tokens.

##### Q3:  À quoi sert la connexion résiduelle?

- La connexion résiduelle permet d'ajouter la correction calculée (par l'attention ou le MLP) au vecteur d'origine plutôt que de le remplacer. Cela préserve l'information initiale à travers les couches et facilite la propagation du gradient pendant l'entraînement. En effet, sa rendre les réseaux profonds plus faciles à entraîner.


## TP 4 - Assembler et entraîner le modèle complet

### Étape 0: préparer le notebook

In [ ]:
batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

assert n_embd % n_head == 0
print(device)

cpu


## Questions d’interprétation

##### Q1: Calculez dh = nemb/nhead.

- d_h = n_embd / n_head = 64 / 4 = 16.

##### Q2: Combien de tokens contient un batch?

- Un batch contient batch_size × block_size = 16 × 32 = 512 tokens.

##### Q3: Sur CPU, quelle variable réduiriez-vous d’abord pour tester?

- Je réduirais d'abord max_iters juste pour vérifier rapidement que le pipeline fonctionne sans erreur avant de lancer un entraînement complet plus long.

### Étape 1: coder une tête

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            'tril', torch.tril(torch.ones(block_size, block_size))
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.size(-1)**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

## Questions d’interprétation

##### Q1: Pourquoi utiliser k.size(-1) plutôt que (C) pour le facteur d’échelle?

- On utilise -1 car dans une attention multi-têtes, les vecteurs Key ont pour dimension head_size qui est plus petite que C. Le facteur d'échelle doit correspondre à la vraie dimension des vecteurs comparés et pas à la dimension d'embedding complète.

##### Q2: Pourquoi tronquer tril[:T,:T]?

- On tronque tril[:T,:T] car ce masque est créé une seule fois à la taille maximale. Mais la séquence traitée peut avoir une longueur T plus courte. Il faut donc adapter la taille du masque à la longueur réelle de la séquence en cours.


###  Étape 2: combiner plusieurs têtes

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat(
            [h(x) for h in self.heads], dim=-1
        )
        return self.dropout(self.proj(out))

## Questions d’interprétation

##### Q1: Pourquoi concaténer sur la dernière dimension?

- On concatène sur la dernière dimension car c'est la dimension des features.

##### Q2: Quel problème aurait-on si nemb n’était pas divisible par nhead?

- Si n_embd n'était pas divisible par n_head, les têtes ne pourraient pas se partager n_embd en portions égales de head_size. La concaténation des sorties ne redonnerait donc pas exactement n_embd dimensions, ce qui empêcherait l'addition résiduelle avec l'entrée d'origine.


###  Étape 3: assembler un bloc

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## Questions d’interprétation

##### Dans quel ordre appliquer LayerNorm, attention/MLP et addition?

- On applique d'abord LayerNorm, puis l'attention ou le MLP sur le résultat normalisé. Puis on ajoute ce résultat à l'entrée d'origine non normalisée via la connexion résiduelle.

##### À quelle opération correspondent « communication » et « calcul »?

- Communication correspond à l'attention, car c'est la seule opération qui permet aux tokens d'échanger de l'information entre eux. Calcul correspond au MLP, car il traite chaque token indépendamment sans interaction entre les positions.


### Étape 4: définir les couches du modèle

###  Étape 5: écrire le passage avant et la loss

### Étape 6: générer avec un contexte limité


In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )
        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head=n_head)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)              #(B,T,n_embd)
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding_table(pos)            #(T,n_embd)

        x = tok_emb + pos_emb                                    #(B,T,n_embd)
        x = self.blocks(x)                                       #(B,T,n_embd)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                 #(B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            logits = logits.reshape(B*T, vocab_size)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)

            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(
                probs, num_samples=1
            )
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Questions d’interprétation

### Étape 4
##### Q1: Donnez les dimensions des trois matrices d’embedding/unembedding.

- token_embedding_table a pour dimensions (vocab_size, n_embd), position_embedding_table a pour dimensions (block_size, n_embd), et lm_head a pour dimensions (n_embd, vocab_size).

##### Q2: Pourquoi lm_head n’est-il plus confondu avec l’embedding ?

- Contrairement au TP1, il y a maintenant de vrais calculs (attention et MLP) entre l'embedding et la prédiction finale. Le token embedding donne une représentation initiale du caractère tandis que lm_head convertit la représentation finale, déjà contextualisée, en logits sur le vocabulaire. Ce sont deux transformations apprises séparément, avec des rôles différents.

###  Étape 5
##### Q3: Quelle règle de broadcasting permet l’addition des embeddings?

- tok_emb a pour forme (B, T, n_embd) et pos_emb a pour forme (T, n_embd). Le broadcasting de PyTorch aligne les dimensions à partir de la droite : les deux dernières dimensions correspondent déjà, et la dimension B manquante dans pos_emb est automatiquement répétée pour chaque séquence du batch.

##### Q4: Pourquoi ne pas aplatir les logits lors de la génération?

- Aplatir les logits n'est utile que pour calculer la loss avec cross_entropy, qui attend une forme (B*T, vocab_size). Pendant la génération, il n'y a pas de cible ni de loss à calculer : on a seulement besoin des logits de la dernière position, donc garder la forme (B, T, vocab_size) et ne sélectionner que le dernier pas de temps est plus simple.

### Étape 6
##### Q5: Pourquoi @torch.no_grad() ?

- @torch.no_grad() désactive le calcul des gradients. Cela économise de la mémoire et du calcul.

##### Q6: Pourquoi échantillonner avec multinomial plutôt que prendre systématiquement argmax ?

- argmax choisirait toujours le caractère le plus probable. Ce qui donnerait un texte déterministe et répétitif. multinomial introduit de la variété en tirant un caractère selon la distribution de probabilités.




### Étapes 7 et 8: évaluer puis entraîner

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate
)
for iteration in range(max_iters):
    if iteration % eval_interval == 0:
        print(estimate_loss())

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

{'train': tensor(3.9872), 'val': tensor(4.0214)}
{'train': tensor(3.8818), 'val': tensor(3.9218)}
{'train': tensor(3.7868), 'val': tensor(3.8222)}
{'train': tensor(3.6987), 'val': tensor(3.7335)}
{'train': tensor(3.6110), 'val': tensor(3.6462)}
{'train': tensor(3.5278), 'val': tensor(3.5639)}
{'train': tensor(3.4531), 'val': tensor(3.4857)}
{'train': tensor(3.3900), 'val': tensor(3.4174)}
{'train': tensor(3.3176), 'val': tensor(3.3500)}
{'train': tensor(3.2571), 'val': tensor(3.2914)}
{'train': tensor(3.1983), 'val': tensor(3.2305)}
{'train': tensor(3.1422), 'val': tensor(3.1741)}
{'train': tensor(3.0939), 'val': tensor(3.1268)}
{'train': tensor(3.0499), 'val': tensor(3.0751)}
{'train': tensor(2.9993), 'val': tensor(3.0349)}
{'train': tensor(2.9684), 'val': tensor(2.9944)}
{'train': tensor(2.9310), 'val': tensor(2.9584)}
{'train': tensor(2.8917), 'val': tensor(2.9315)}
{'train': tensor(2.8637), 'val': tensor(2.8921)}
{'train': tensor(2.8368), 'val': tensor(2.8670)}
{'train': tensor(2.8

## Questions d’interprétation

##### Q1: Pourquoi appeler model.eval() puis model.train()?

- model.eval() met le modèle en mode évaluation. C'est important pour des couches comme Dropout qui se comportent différemment à l'entraînement et à l'inférence. model.train() remet ensuite le modèle en mode entraînement pour que les étapes suivantes se comportent normalement.

#### Q2: Si la loss train baisse mais que la loss validation remonte, que se passe-t-il?

- Le modèle sur-apprend: il mémorise de plus en plus les données d'entraînement au lieu d'apprendre des régularités généralisables. En effet, ca améliore son score sur le train mais dégrade sa performance sur des données non vues.

### Étape 9: entraîner, générer et expérimenter

In [ ]:
model = BigramLanguageModel().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for iteration in range(max_iters):
    if iteration % eval_interval == 0:
        print(estimate_loss())
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
result = model.generate(context, max_new_tokens=2000)
print(decode(result[0].tolist()))

0.209729 M parameters
{'train': tensor(4.2843), 'val': tensor(4.2976)}
{'train': tensor(2.6742), 'val': tensor(2.6878)}
{'train': tensor(2.4960), 'val': tensor(2.5115)}
{'train': tensor(2.3980), 'val': tensor(2.3978)}
{'train': tensor(2.3245), 'val': tensor(2.3323)}
{'train': tensor(2.2721), 'val': tensor(2.2801)}
{'train': tensor(2.2246), 'val': tensor(2.2392)}
{'train': tensor(2.1664), 'val': tensor(2.1821)}
{'train': tensor(2.1235), 'val': tensor(2.1488)}
{'train': tensor(2.0985), 'val': tensor(2.1191)}
{'train': tensor(2.0469), 'val': tensor(2.1032)}
{'train': tensor(2.0308), 'val': tensor(2.0871)}
{'train': tensor(2.0024), 'val': tensor(2.0752)}
{'train': tensor(1.9890), 'val': tensor(2.0604)}
{'train': tensor(1.9621), 'val': tensor(2.0382)}
{'train': tensor(1.9383), 'val': tensor(2.0165)}
{'train': tensor(1.9179), 'val': tensor(1.9965)}
{'train': tensor(1.9037), 'val': tensor(1.9956)}
{'train': tensor(1.8920), 'val': tensor(1.9885)}
{'train': tensor(1.8583), 'val': tensor(1.9801)

## Bilan des quatre TP

- Le token embedding et le position embedding créent le contexte initial.

- L'attention (Head, MultiHeadAttention) transforme chaque token en mélangeant l'information des tokens précédents.

- Le MLP (FeedForward) transforme chaque token indépendamment. Le lm_head produit les logits finaux sur le vocabulaire.

- La loss agit sur l'ensemble du modèle.

- La rétropropagation fait remonter le gradient à travers lm_head, les blocs, jusqu'aux embeddings, ajustant tous les paramètres pour réduire l'erreur de prédiction.